In [ ]:
!pip -q install insightface onnxruntime opencv-python pillow matplotlib numpy > /dev/null

In [ ]:
import io, math, time
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display, Javascript
from google.colab import files


In [ ]:
from insightface.app import FaceAnalysis
app = FaceAnalysis(name="buffalo_l", providers=["CPUExecutionProvider"])
app.prepare(ctx_id=-1, det_size=(640, 640))

print("InsightFace loaded ✅")

In [ ]:
def pil_to_bgr(pil_img: Image.Image) -> np.ndarray:
    rgb = np.array(pil_img.convert("RGB"))
    return cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)

def best_face(faces):
    if not faces:
        return None
    areas = []
    for f in faces:
        x1, y1, x2, y2 = f.bbox
        areas.append((x2-x1) * (y2-y1))
    return faces[int(np.argmax(areas))]

def l2_normalize(v: np.ndarray, eps=1e-12):
    n = np.linalg.norm(v) + eps
    return v / n

def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    a = l2_normalize(a)
    b = l2_normalize(b)
    return float(np.dot(a, b))

def get_embedding_from_bgr(bgr: np.ndarray, det_thresh=0.60):
    faces = app.get(bgr)
    f = best_face(faces)
    if f is None:
        return None, None, None
    if float(f.det_score) < det_thresh:
        return None, None, None
    emb = np.array(f.embedding, dtype=np.float32)
    return emb, np.array(f.bbox, dtype=np.float32), float(f.det_score)


In [ ]:
print("Upload EXACTLY 6 images of the same person (jpg/png).")
uploaded = files.upload()

file_list = list(uploaded.keys())
print("Uploaded files:", file_list)

if len(file_list) != 6:
    raise ValueError(f"You must upload exactly 6 images. You uploaded {len(file_list)}.")


In [ ]:
DET_THRESH_ENROLL = 0.60
MIN_SIM_DEBUG = True

enroll_embeddings = []
bad_files = []

for fn in file_list:
    pil_img = Image.open(io.BytesIO(uploaded[fn]))
    bgr = pil_to_bgr(pil_img)

    emb, bbox, score = get_embedding_from_bgr(bgr, det_thresh=DET_THRESH_ENROLL)
    if emb is None:
        bad_files.append(fn)
        continue

    enroll_embeddings.append(l2_normalize(emb))

if bad_files:
    raise ValueError(
        "Face not detected confidently in these images (re-upload clearer face shots):\n"
        + "\n".join(bad_files)
    )

enroll_embeddings = np.stack(enroll_embeddings, axis=0)
template = l2_normalize(np.mean(enroll_embeddings, axis=0))

print("Enrollment complete ✅")
print("Template embedding shape:", template.shape)


In [ ]:
sims = []
for i in range(6):
    sims.append(cosine_sim(enroll_embeddings[i], template))

print("Enrollment similarity to template (higher is better):")
for i, (fn, s) in enumerate(zip(file_list, sims)):
    print(f"{i+1}. {fn}: {s:.4f}")

print("Mean sim:", float(np.mean(sims)))
print("Min  sim:", float(np.min(sims)))


In [ ]:
import base64
def capture_frame_from_webcam(width=640, height=480, quality=0.9):
    js = Javascript(f"""
    async function captureFrame() {{
      const div = document.createElement('div');
      const video = document.createElement('video');
      video.style.border = '2px solid black';
      video.width = {width};
      video.height = {height};
      div.appendChild(video);

      const stream = await navigator.mediaDevices.getUserMedia({{video: true}});
      document.body.appendChild(div);
      video.srcObject = stream;
      await video.play();
      await new Promise((resolve) => setTimeout(resolve, 700));

      const canvas = document.createElement('canvas');
      canvas.width = {width};
      canvas.height = {height};
      canvas.getContext('2d').drawImage(video, 0, 0, canvas.width, canvas.height);
      stream.getTracks().forEach(track => track.stop());
      div.remove();

      return canvas.toDataURL('image/jpeg', {quality});
    }}
    captureFrame();
    """)
    display(js)

    from google.colab import output
    data_url = output.eval_js("captureFrame()")
    header, b64data = data_url.split(",", 1)
    img_bytes = io.BytesIO(base64.b64decode(b64data))
    pil_img = Image.open(img_bytes)
    return pil_img


In [ ]:
LIVE_FRAMES = 5
DET_THRESH_LIVE = 0.60
SIM_THRESHOLD = 0.35

live_sims = []
live_scores = []
no_face_frames = 0

print("Webcam will turn on multiple times to capture frames. Please look at the camera.")
for i in range(LIVE_FRAMES):
    print(f"\nCapturing frame {i+1}/{LIVE_FRAMES} ...")
    pil_live = capture_frame_from_webcam(width=640, height=480, quality=0.92)

    bgr_live = pil_to_bgr(pil_live)
    emb_live, bbox_live, det_score = get_embedding_from_bgr(bgr_live, det_thresh=DET_THRESH_LIVE)

    if emb_live is None:
        no_face_frames += 1
        print("No confident face detected in this frame.")
        continue

    sim = cosine_sim(emb_live, template)
    live_sims.append(sim)
    live_scores.append(det_score)
    print(f"Detected face score={det_score:.3f} | similarity={sim:.4f}")

if len(live_sims) == 0:
    decision = "NO MATCH"
    reason = f"No confident face detected in any live frame ({no_face_frames}/{LIVE_FRAMES})."
else:
    best_sim = float(np.max(live_sims))
    mean_sim = float(np.mean(live_sims))
    decision = "MATCH" if best_sim >= SIM_THRESHOLD else "NO MATCH"
    reason = f"best_sim={best_sim:.4f}, mean_sim={mean_sim:.4f}, threshold={SIM_THRESHOLD:.2f}"

print("\n========================")
print("AI Live Selfie Check ")
print("FINAL OUTCOME:", decision)
print("DETAILS:", reason)
print("========================")


In [ ]:
plt.figure(figsize=(6,4))
plt.imshow(np.array(pil_live.convert("RGB")))
plt.axis("off")
plt.title("Last captured live frame")
plt.show()
